In [0]:
from pyspark.sql.functions import rand, when, col
from pyspark.sql import functions as F
 
# Create base DataFrame with 1000 rows
df = spark.range(1, 1001).withColumnRenamed("id", "order_id")
 
# Add columns
df = df.withColumn("customer_id", (rand()*100).cast("int")) \
       .withColumn("age", (rand()*50 + 20).cast("int")) \
       .withColumn("gender", when(rand() > 0.5, "Male").otherwise("Female")) \
       .withColumn("product_category",
                   when(rand() < 0.3, "Electronics")
                   .when(rand() < 0.6, "Clothing")
                   .otherwise("Grocery")) \
       .withColumn("city",
                   when(rand() < 0.25, "Kochi")
                   .when(rand() < 0.5, "Bangalore")
                   .when(rand() < 0.75, "Chennai")
                   .otherwise("Hyderabad")) \
       .withColumn("quantity", (rand()*5 + 1).cast("int")) \
       .withColumn("price", (rand()*5000 + 100).cast("int")) \
       .withColumn("order_type",
                   when(rand() > 0.5, "Online").otherwise("Offline"))
 
# Revenue column
df = df.withColumn("total_amount", col("quantity") * col("price"))
 
df.show()

+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+
|order_id|customer_id|age|gender|product_category|     city|quantity|price|order_type|total_amount|
+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+
|       1|         72| 52|  Male|        Clothing|Bangalore|       5| 2946|    Online|       14730|
|       2|         32| 33|  Male|        Clothing|  Chennai|       5| 1177|    Online|        5885|
|       3|         73| 32|  Male|         Grocery|  Chennai|       5| 1897|    Online|        9485|
|       4|         16| 23|Female|     Electronics|  Chennai|       2| 3332|    Online|        6664|
|       5|         96| 69|  Male|     Electronics|    Kochi|       2|  774|   Offline|        1548|
|       6|         14| 51|Female|     Electronics|    Kochi|       3| 2961|    Online|        8883|
|       7|         64| 30|Female|     Electronics|Bangalore|       2| 4042|   Offline|        8084|


In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = false)
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- product_category: string (nullable = false)
 |-- city: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_type: string (nullable = false)
 |-- total_amount: integer (nullable = true)



# Data Transformation

## Create New Columns

### 1. age_group
- < 30   → Young  
- 30–50  → Adult  
- > 50   → Senior  

 

In [0]:
from pyspark.sql.functions import when
df.withColumn("age_group",when(df.age < 30,"Young").when((df.age >= 30) &(df.age <= 50),"Adult").otherwise("Senior")).show()

+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+---------+
|order_id|customer_id|age|gender|product_category|     city|quantity|price|order_type|total_amount|age_group|
+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+---------+
|       1|         72| 52|  Male|        Clothing|Bangalore|       5| 2946|    Online|       14730|   Senior|
|       2|         32| 33|  Male|        Clothing|  Chennai|       5| 1177|    Online|        5885|    Adult|
|       3|         73| 32|  Male|         Grocery|  Chennai|       5| 1897|    Online|        9485|    Adult|
|       4|         16| 23|Female|     Electronics|  Chennai|       2| 3332|    Online|        6664|    Young|
|       5|         96| 69|  Male|     Electronics|    Kochi|       2|  774|   Offline|        1548|   Senior|
|       6|         14| 51|Female|     Electronics|    Kochi|       3| 2961|    Online|        8883|   Senior|
|       7|

### 2. revenue_category
- total_amount < 2000   → Low  
- 2000 – 7000           → Medium  
- > 7000                → High 

In [0]:
df.withColumn("revenue_category",when(df.total_amount < 2000,"Low").when((df.total_amount >= 2000) & (df.total_amount <= 7000),"Medium").otherwise("High")).show()

+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+----------------+
|order_id|customer_id|age|gender|product_category|     city|quantity|price|order_type|total_amount|revenue_category|
+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+----------------+
|       1|         72| 52|  Male|        Clothing|Bangalore|       5| 2946|    Online|       14730|            High|
|       2|         32| 33|  Male|        Clothing|  Chennai|       5| 1177|    Online|        5885|          Medium|
|       3|         73| 32|  Male|         Grocery|  Chennai|       5| 1897|    Online|        9485|            High|
|       4|         16| 23|Female|     Electronics|  Chennai|       2| 3332|    Online|        6664|          Medium|
|       5|         96| 69|  Male|     Electronics|    Kochi|       2|  774|   Offline|        1548|             Low|
|       6|         14| 51|Female|     Electronics|    Kochi|    

# Analysis Questions

## 🔹 Task 1
- Calculate total sales (revenue) by `product_category`

In [0]:
from pyspark.sql.functions import sum
df.groupBy("product_category").agg(sum("total_amount").alias("total_sales")).show()

+----------------+-----------+
|product_category|total_sales|
+----------------+-----------+
|         Grocery|    2256525|
|     Electronics|    2551684|
|        Clothing|    2930437|
+----------------+-----------+



## 🔹 Task 2
- Count number of orders by `city`

In [0]:
from pyspark.sql.functions import count
df.groupBy("city").agg(count("*").alias("no_of_orders")).show()

+---------+------------+
|     city|no_of_orders|
+---------+------------+
|  Chennai|         291|
|    Kochi|         247|
|Hyderabad|         100|
|Bangalore|         362|
+---------+------------+



## 🔹 Task 3
- Compute average order value by `order_type`

In [0]:
from pyspark.sql.functions import avg
df.groupBy("order_type").agg(avg("total_amount").alias("order_value")).display()

order_type,order_value
Online,7778.178502879079
Offline,7695.647181628393


## 🔹 Task 4
- Identify top 3 cities by total revenue

In [0]:
from pyspark.sql.functions import desc
df.groupBy("city").agg(sum("total_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(3)

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|      2773791|
|  Chennai|      2108096|
|    Kochi|      2032969|
+---------+-------------+
only showing top 3 rows


## 🔹 Task 5
- Find revenue contribution by `product_category`

In [0]:
print(df.agg(sum("total_amount")).first()[0])
total = df.agg(sum("total_amount")).first()[0]
df.groupBy("product_category").agg((sum("total_amount"))).show()
df.groupBy("product_category").agg(((sum("total_amount")/total)*100).alias("revenue_percentage")).show()

7738646
+----------------+-----------------+
|product_category|sum(total_amount)|
+----------------+-----------------+
|         Grocery|          2256525|
|     Electronics|          2551684|
|        Clothing|          2930437|
+----------------+-----------------+

+----------------+------------------+
|product_category|revenue_percentage|
+----------------+------------------+
|         Grocery|29.159170738653767|
|     Electronics|  32.9732617307989|
|        Clothing| 37.86756753054733|
+----------------+------------------+



## 🔹 Task 6
For each `product_category`, calculate:
- Total number of orders  
- Total revenue  

In [0]:
df.groupBy("product_category").agg(count("order_id").alias("no_of_orders"), sum("total_amount").alias("total_revenue")).show()

+----------------+------------+-------------+
|product_category|no_of_orders|total_revenue|
+----------------+------------+-------------+
|         Grocery|         282|      2256525|
|     Electronics|         314|      2551684|
|        Clothing|         404|      2930437|
+----------------+------------+-------------+



## 🔹 Task 7
- Count of high revenue orders (> 7000) per city


In [0]:
df.groupBy("city").agg(count(when(df.total_amount > 7000, 1)).alias("high_revenue")).show()

+---------+------------+
|     city|high_revenue|
+---------+------------+
|  Chennai|         121|
|    Kochi|         118|
|Hyderabad|          50|
|Bangalore|         169|
+---------+------------+



In [0]:
df.filter(df.total_amount > 7000).groupBy("city").agg(count("*").alias("high_revenue")).show()

+---------+------------+
|     city|high_revenue|
+---------+------------+
|  Chennai|         121|
|    Kochi|         118|
|Hyderabad|          50|
|Bangalore|         169|
+---------+------------+



## 🔹 Task 8
- Identify which `product_category` generates the highest revenue

In [0]:
df.groupBy("product_category").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+----------------+-------+
|product_category|revenue|
+----------------+-------+
|        Clothing|2930437|
+----------------+-------+
only showing top 1 row


## 🔹 Task 9
- Find the city with the highest average order value

In [0]:
df.groupBy("city").agg(avg("total_amount").alias("order_value")).orderBy(desc("order_value")).show(1)


+---------+-----------+
|     city|order_value|
+---------+-----------+
|Hyderabad|     8237.9|
+---------+-----------+
only showing top 1 row


## 🔹 Task 10
- Identify top `product_category` in each city based on revenue

In [0]:
T = df.groupBy("city", "product_category").agg(sum("total_amount").alias("revenue")).show()

+---------+----------------+-------+
|     city|product_category|revenue|
+---------+----------------+-------+
|    Kochi|        Clothing| 785938|
|Bangalore|         Grocery| 840973|
|Hyderabad|         Grocery| 148851|
|Hyderabad|     Electronics| 264234|
|Hyderabad|        Clothing| 410705|
|Bangalore|        Clothing|1089780|
|  Chennai|     Electronics| 835283|
|    Kochi|         Grocery| 637902|
|  Chennai|         Grocery| 628799|
|    Kochi|     Electronics| 609129|
|  Chennai|        Clothing| 644014|
|Bangalore|     Electronics| 843038|
+---------+----------------+-------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("city").orderBy(desc("total_revenue"))

df_revenue = df.groupBy("city", "product_category").agg(sum("total_amount").alias("total_revenue"))
top_categories = df_revenue.withColumn("rank", row_number().over(window_spec)).filter("rank = 1")
display(top_categories.select("city", "product_category", "total_revenue"))

city,product_category,total_revenue
Bangalore,Clothing,1089780
Chennai,Electronics,835283
Hyderabad,Clothing,410705
Kochi,Clothing,785938


In [0]:
cities = [row["city"] for row in df.select("city").distinct().collect()]
for city in cities:
    citydf = df.filter(df.city == city)
    # citydf.show()
    group = citydf.groupBy("product_category").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue"))
    group.show()
    print(city, group.first()[0])

+----------------+-------+
|product_category|revenue|
+----------------+-------+
|     Electronics| 835283|
|        Clothing| 644014|
|         Grocery| 628799|
+----------------+-------+

Chennai Electronics
+----------------+-------+
|product_category|revenue|
+----------------+-------+
|        Clothing| 785938|
|         Grocery| 637902|
|     Electronics| 609129|
+----------------+-------+

Kochi Clothing
+----------------+-------+
|product_category|revenue|
+----------------+-------+
|        Clothing| 410705|
|     Electronics| 264234|
|         Grocery| 148851|
+----------------+-------+

Hyderabad Clothing
+----------------+-------+
|product_category|revenue|
+----------------+-------+
|        Clothing|1089780|
|     Electronics| 843038|
|         Grocery| 840973|
+----------------+-------+

Bangalore Clothing


# Insight Questions
- Q1 Which category is most profitable?

In [0]:
df.groupBy("product_category").agg(sum("total_amount").alias("profitable")).orderBy(desc("profitable")).first()[0]

'Clothing'

- Q2 Which city is performing best?

In [0]:
df.groupBy("city").agg(sum("total_amount").alias("performance")).orderBy(desc("performance")).first()[0]

'Bangalore'

- Q3 Are online or offline orders generating more revenue?

In [0]:
df.groupBy("order_type").agg(sum("total_amount").alias("revenue")).orderBy(desc("revenue")).first()[0]

'Online'

- Q4 Which age group contributes most revenue?

In [0]:
from pyspark.sql.functions import count,desc,sum
newdf = df.withColumn("age_group",when(df.age <= 20, "0-20").when((df.age > 20) & (df.age <= 30), "21-30").when((df.age > 30) & (df.age <= 40), "31-40").when((df.age > 40) & (df.age <= 50), "41-50").when((df.age > 50) & (df.age <= 60), "51-60").when(df.age > 60, "60+").otherwise("Unknown"))
newdf.show()
newdf.groupBy("age_group").agg(sum("total_amount").alias("total_amount")).orderBy(desc("total_amount")).first()[0]



+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+---------+
|order_id|customer_id|age|gender|product_category|     city|quantity|price|order_type|total_amount|age_group|
+--------+-----------+---+------+----------------+---------+--------+-----+----------+------------+---------+
|       1|         72| 52|  Male|        Clothing|Bangalore|       5| 2946|    Online|       14730|    51-60|
|       2|         32| 33|  Male|        Clothing|  Chennai|       5| 1177|    Online|        5885|    31-40|
|       3|         73| 32|  Male|         Grocery|  Chennai|       5| 1897|    Online|        9485|    31-40|
|       4|         16| 23|Female|     Electronics|  Chennai|       2| 3332|    Online|        6664|    21-30|
|       5|         96| 69|  Male|     Electronics|    Kochi|       2|  774|   Offline|        1548|      60+|
|       6|         14| 51|Female|     Electronics|    Kochi|       3| 2961|    Online|        8883|    51-60|
|       7|

'31-40'